# 3W Coverage

**Project:** Pali Watch

**Notebook:** `3w_coverage.ipynb`

**Author:** Yuseof J

**Created:** 11/11/2026 | **Last updated:** 14/11/2026

## Purpose
This file presents a more coarse historical view of org coverage in Palestine

## Inputs

- `data/raw/boundaries/`: HDX admin boundaries (COD-AB)

## Outputs
- `data/processed/[filename].csv`: what it contains

## Notes and caveats
- Data limitations: data source ends in April 2023, so its completely obsolete for Gaza, and for the West Bank provides only a historical snapshot. This is more of a placeholder in lieu of more recent 3w data

In [ ]:
import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

CRS = 'EPSG:4326'

## Data imports / exports

In [ ]:
# import paths
PATH_TO_ADMIN_BOUNDARIES = f'{RAW}/pse_admin_boundaries.shp/pse_admin2.shp'
PATH_TO_3W_COVERAGE = f'{RAW}/3w-coverage/pse-who-what-where-3w.xlsx'

# export paths
PATH_TO_GEOLOCATED_EVENTS_GPKG = f'{PROCESSED}/acled/conflict_events_geolocated.gpkg'
PATH_TO_GEOLOCATED_EVENTS_DF = f'{PROCESSED}/acled/conflict_events_geolocated.csv'

# import data
df_3w = pd.read_excel(PATH_TO_3W_COVERAGE)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

## Preprocessing

In [ ]:
# remove second row 
if df_3w.iloc[0].Cluster == "#sector+name":
    df_3w.drop([0], inplace=True)

In [ ]:
# ensure matching governorate names
set_boundary_govs = set(gdf_boundaries.adm2_name.unique())
set_3w_govs = set(df_3w.Governorates.unique())
set_unaccounted = set_boundary_govs - set_3w_govs
print(f"The following govs are mismatched/unaccounted for: {set_unaccounted}")

dict_unaccounted_govs = {
    'Gaza North': 'North Gaza', 
    'Qalqiliya': 'Qalqilya', 
    'Khan Yunis': 'Khan Younis', 
    'Deir Al-Balah (Middle area)': 'Deir Al-Balah' 
}

# ensure all mismatched gov names are accounted for
set_unaccounted - set(dict_unaccounted_govs.values())
assert(set_unaccounted - set(dict_unaccounted_govs.values()) == set())
print("All governorates successfully updated")

# replace mismatched gov names
df_3w.replace(dict_unaccounted_govs, inplace=True)

## Governorate Aggregates

In [ ]:
df_3w.head()

In [ ]:
df_3w.Cluster.unique()

#### Org Count

In [ ]:
df_3w.groupby(['Governorates']).Organization.nunique()

#### Clusters covered

In [ ]:
df_3w.groupby(['Governorates']).Cluster.nunique()

#### Orgs per cluster

In [ ]:
df_3w.groupby(['Governorates', 'Cluster']).Organization.nunique()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate data
plot_df = (
    df_3w.groupby(['Governorates', 'Cluster'])['Organization']
    .nunique()
    .reset_index(name='Unique Organizations')
)

# Plot
plt.figure(figsize=(16, 7))

sns.barplot(
    data=plot_df,
    x='Governorates',
    y='Unique Organizations',
    hue='Cluster',
    palette='Set2'
)

plt.xlabel('Governorate')
plt.ylabel('Number of Unique Organizations')
plt.title('Number of Unique Organizations by Governorate and Cluster')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Mean Orgs Per Cluster (Across Govs)

In [ ]:
mean_orgs = (
    df_3w.groupby(['Governorates', 'Cluster'])['Organization']
    .nunique()
    .groupby('Cluster')
    .mean()
    .reset_index(name='Mean Organizations')
)

print(mean_orgs)

## Geoprocessing

## Export Results